In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd

In [ ]:
# Pandas :: view ALL
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Pandas :: Default view
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [5]:
# Defines
def analyse_dataframe(df):
    nan_cnt = df.isnull().sum()
    nan_rto = df.isnull().mean()*100
    unique_vals_cnt = df.nunique()

    ana_result = pd.DataFrame({
        'NaN Counts' : nan_cnt
        , 'NaN Ratio (%)' : nan_rto
        , 'Unique Values' : unique_vals_cnt
    })
    return ana_result


In [6]:
# PRE

TXT_DIR  = r"C:\Users\user\Documents\MATLAB\wfu_results\ROI_txt"
NII_DIR  = r"D:\JHAhn\8641_conductivity\8641_CDT-CDT_ex-CDT_in\CDT_unZip\HFC"

ORGANS_ORDER = [
    "Amygdala", "Hippocampus", "Insula", "MTG",
    "ParaHippocampus", "Thalamus", "CC", "Precuneus"
]
SIDE_ORDER = ["LT", "RT", "Whole"]
COND_ORDER = ["HFC", "IC", "EC"]

In [7]:
pattern = re.compile(r"^(HFC|IC|EC)_(LT|RT|Whole)_(.+)\.txt$", re.I)
col_data = {}   # {col_name: Series}

for fp in glob.glob(os.path.join(TXT_DIR, "*.txt")):
    name = os.path.basename(fp)
    m = pattern.match(name)
    if not m:
        raise ValueError(f"파일명 패턴 불일치: {name}")

    cond, side, organ = m.group(1), m.group(2), m.group(3)
    cond, side = cond.upper(), side  # cond만 대문자로 고정
    organ = organ.replace(" ", "")   # 공백이 있으면 제거

    # 274행 검증
    s = pd.read_csv(fp, header=None).squeeze("columns")
    if len(s) != 274:
        raise ValueError(f"행 수(274) 불일치: {name} → {len(s)}행")

    col_name = f"{cond}_{side}_{organ}"
    col_data[col_name] = s

df = pd.DataFrame(col_data)
col_data

ValueError: 행 수(274) 불일치: EC_LT_Amygdala.txt → 10행

In [12]:
expected_cols = []
for side in SIDE_ORDER:
    for organ in ORGANS_ORDER:
        for cond in COND_ORDER:
            cname = f"{cond}_{side}_{organ}"
            if cname in df.columns:
                expected_cols.append(cname)

# 누락 또는 여분 점검
missing = [c for c in expected_cols if c not in df.columns]
extra   = [c for c in df.columns if c not in expected_cols]
if missing:
    raise ValueError(f": {missing}")
if extra:
    expected_cols += extra

df = df[expected_cols]
df

,HFC_LT_Amygdala,IC_LT_Amygdala,EC_LT_Amygdala,HFC_LT_Hippocampus,IC_LT_Hippocampus,EC_LT_Hippocampus,HFC_LT_Insula,IC_LT_Insula,EC_LT_Insula,HFC_LT_MTG,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,4.9788860e-01,0.061961,0.410967,0.838401,0.056949,0.556093,0.569074,0.087700,0.498727,0.566726,...,0.543074,0.826938,0.156422,0.670541,0.566084,0.121781,0.444449,0.709944,0.084998,0.624102
1,6.4992461e-01,0.114312,0.539604,0.783283,0.074173,0.619056,0.621108,0.132052,0.516985,0.621009,...,0.556320,1.045575,0.328254,0.717209,0.593882,0.157959,0.435958,0.751718,0.087023,0.667197
2,6.3518651e-01,0.055755,0.537469,0.724749,0.069935,0.650119,0.543414,0.061849,0.511009,0.572327,...,0.560632,1.030501,0.214016,0.816450,0.518856,0.102770,0.416148,0.672717,0.084116,0.586778
3,7.6429523e-01,0.071338,0.592826,0.843046,0.061556,0.595581,0.627041,0.086411,0.603624,0.610571,...,0.610830,0.999321,0.200949,0.798366,0.573783,0.099075,0.474813,0.730034,0.078498,0.647279
4,6.6688490e-01,0.068983,0.614509,0.975962,0.054164,0.751326,0.623509,0.064267,0.627291,0.711997,...,0.602847,0.991634,0.190448,0.801202,0.567557,0.126154,0.441525,0.791421,0.066190,0.727058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,5.6422101e-01,0.077352,0.523213,0.666636,0.090259,0.581154,0.528409,0.100684,0.451361,0.524941,...,0.474700,0.901093,0.220515,0.680470,0.488034,0.144081,0.344001,0.621078,0.117980,0.506970
270,6.9848789e-01,0.133280,0.594502,0.743387,0.105156,0.491800,0.542317,0.154168,0.442211,0.481155,...,0.526787,0.860828,0.222414,0.638332,0.536061,0.155244,0.380953,0.602585,0.118519,0.487089
271,8.3637830e-01,0.075306,0.682050,0.715348,0.075344,0.519286,0.524379,0.093592,0.488558,0.514201,...,0.343942,0.769077,0.179261,0.590332,0.535888,0.157677,0.378221,0.599822,0.092429,0.508567
272,4.9683935e-01,0.090352,0.425914,0.759475,0.112112,0.489371,0.569115,0.132434,0.478665,0.560055,...,0.368515,0.890930,0.280834,0.610054,0.601955,0.208697,0.393320,0.717549,0.149375,0.571504


In [18]:
subject_files = sorted(glob.glob(os.path.join(NII_DIR, "wConduct_*.nii")))
if len(subject_files) != len(df):
    raise ValueError(f"행 수(274)와 .nii 파일 수({len(subject_files)}) 불일치")

extract_code = lambda path: os.path.splitext(os.path.basename(path))[0].split("_", 1)[1]
df.index = [extract_code(p) for p in subject_files]
df.index.name = "PDI"

df

,HFC_LT_Amygdala,IC_LT_Amygdala,EC_LT_Amygdala,HFC_LT_Hippocampus,IC_LT_Hippocampus,EC_LT_Hippocampus,HFC_LT_Insula,IC_LT_Insula,EC_LT_Insula,HFC_LT_MTG,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
PDI,,,,,,,,,,,,,,,,,,,,,
ADEPT01_YSJ_NODDI24,4.9788860e-01,0.061961,0.410967,0.838401,0.056949,0.556093,0.569074,0.087700,0.498727,0.566726,...,0.543074,0.826938,0.156422,0.670541,0.566084,0.121781,0.444449,0.709944,0.084998,0.624102
ADEPT02_LKY_NODDI24,6.4992461e-01,0.114312,0.539604,0.783283,0.074173,0.619056,0.621108,0.132052,0.516985,0.621009,...,0.556320,1.045575,0.328254,0.717209,0.593882,0.157959,0.435958,0.751718,0.087023,0.667197
ADEPT03_YKJ_NODDI24,6.3518651e-01,0.055755,0.537469,0.724749,0.069935,0.650119,0.543414,0.061849,0.511009,0.572327,...,0.560632,1.030501,0.214016,0.816450,0.518856,0.102770,0.416148,0.672717,0.084116,0.586778
ADEPT04_JWS_NODDI24,7.6429523e-01,0.071338,0.592826,0.843046,0.061556,0.595581,0.627041,0.086411,0.603624,0.610571,...,0.610830,0.999321,0.200949,0.798366,0.573783,0.099075,0.474813,0.730034,0.078498,0.647279
ADEPT05_CBK_NODDI24,6.6688490e-01,0.068983,0.614509,0.975962,0.054164,0.751326,0.623509,0.064267,0.627291,0.711997,...,0.602847,0.991634,0.190448,0.801202,0.567557,0.126154,0.441525,0.791421,0.066190,0.727058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MWI_CN82_LMR_NODDI32,5.6422101e-01,0.077352,0.523213,0.666636,0.090259,0.581154,0.528409,0.100684,0.451361,0.524941,...,0.474700,0.901093,0.220515,0.680470,0.488034,0.144081,0.344001,0.621078,0.117980,0.506970
MWI_CN83_LYS_NODDI32,6.9848789e-01,0.133280,0.594502,0.743387,0.105156,0.491800,0.542317,0.154168,0.442211,0.481155,...,0.526787,0.860828,0.222414,0.638332,0.536061,0.155244,0.380953,0.602585,0.118519,0.487089
MWI_CN84_LHS_NODDI32,8.3637830e-01,0.075306,0.682050,0.715348,0.075344,0.519286,0.524379,0.093592,0.488558,0.514201,...,0.343942,0.769077,0.179261,0.590332,0.535888,0.157677,0.378221,0.599822,0.092429,0.508567


In [20]:
df.to_excel(os.path.join(TXT_DIR, "ROI_results.xlsx"), index=True)

In [ ]:
## Data Merging below::

In [36]:
dfROI = pd.read_excel(os.path.join(TXT_DIR, "ROI_results.xlsx"), index_col=0)
dfROI

,HFC_LT_Amygdala,IC_LT_Amygdala,EC_LT_Amygdala,HFC_LT_Hippocampus,IC_LT_Hippocampus,EC_LT_Hippocampus,HFC_LT_Insula,IC_LT_Insula,EC_LT_Insula,HFC_LT_MTG,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
PDI,,,,,,,,,,,,,,,,,,,,,
ADEPT01_YSJ_NODDI24,4.9788860e-01,0.061961,0.410967,0.838401,0.056949,0.556093,0.569074,0.087700,0.498727,0.566726,...,0.543074,0.826938,0.156422,0.670541,0.566084,0.121781,0.444449,0.709944,0.084998,0.624102
ADEPT02_LKY_NODDI24,6.4992461e-01,0.114312,0.539604,0.783283,0.074173,0.619056,0.621108,0.132052,0.516985,0.621009,...,0.556320,1.045575,0.328254,0.717209,0.593882,0.157959,0.435958,0.751718,0.087023,0.667197
ADEPT03_YKJ_NODDI24,6.3518651e-01,0.055755,0.537469,0.724749,0.069935,0.650119,0.543414,0.061849,0.511009,0.572327,...,0.560632,1.030501,0.214016,0.816450,0.518856,0.102770,0.416148,0.672717,0.084116,0.586778
ADEPT04_JWS_NODDI24,7.6429523e-01,0.071338,0.592826,0.843046,0.061556,0.595581,0.627041,0.086411,0.603624,0.610571,...,0.610830,0.999321,0.200949,0.798366,0.573783,0.099075,0.474813,0.730034,0.078498,0.647279
ADEPT05_CBK_NODDI24,6.6688490e-01,0.068983,0.614509,0.975962,0.054164,0.751326,0.623509,0.064267,0.627291,0.711997,...,0.602847,0.991634,0.190448,0.801202,0.567557,0.126154,0.441525,0.791421,0.066190,0.727058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MWI_CN82_LMR_NODDI32,5.6422101e-01,0.077352,0.523213,0.666636,0.090259,0.581154,0.528409,0.100684,0.451361,0.524941,...,0.474700,0.901093,0.220515,0.680470,0.488034,0.144081,0.344001,0.621078,0.117980,0.506970
MWI_CN83_LYS_NODDI32,6.9848789e-01,0.133280,0.594502,0.743387,0.105156,0.491800,0.542317,0.154168,0.442211,0.481155,...,0.526787,0.860828,0.222414,0.638332,0.536061,0.155244,0.380953,0.602585,0.118519,0.487089
MWI_CN84_LHS_NODDI32,8.3637830e-01,0.075306,0.682050,0.715348,0.075344,0.519286,0.524379,0.093592,0.488558,0.514201,...,0.343942,0.769077,0.179261,0.590332,0.535888,0.157677,0.378221,0.599822,0.092429,0.508567


In [38]:
# Extract before 1st '_'

def keep_until_2nd_underscore_from_right(x: object) -> str:
    parts = str(x).split("_")
    return "_".join(parts[:-2]) if len(parts) >= 3 else parts[0]

dfROI["PDI"] = (
    dfROI.index.to_series()    # Index to Type(Series)
         .map(keep_until_2nd_underscore_from_right)
         .astype("string")
)
roi_cols = [c for c in dfROI.columns if c != 'PDI']
dfROI_sub = dfROI.reset_index(drop=True)[['PDI', *roi_cols]]

dfROI_sub

,PDI,HFC_LT_Amygdala,IC_LT_Amygdala,EC_LT_Amygdala,HFC_LT_Hippocampus,IC_LT_Hippocampus,EC_LT_Hippocampus,HFC_LT_Insula,IC_LT_Insula,EC_LT_Insula,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,ADEPT01,4.9788860e-01,0.061961,0.410967,0.838401,0.056949,0.556093,0.569074,0.087700,0.498727,...,0.543074,0.826938,0.156422,0.670541,0.566084,0.121781,0.444449,0.709944,0.084998,0.624102
1,ADEPT02,6.4992461e-01,0.114312,0.539604,0.783283,0.074173,0.619056,0.621108,0.132052,0.516985,...,0.556320,1.045575,0.328254,0.717209,0.593882,0.157959,0.435958,0.751718,0.087023,0.667197
2,ADEPT03,6.3518651e-01,0.055755,0.537469,0.724749,0.069935,0.650119,0.543414,0.061849,0.511009,...,0.560632,1.030501,0.214016,0.816450,0.518856,0.102770,0.416148,0.672717,0.084116,0.586778
3,ADEPT04,7.6429523e-01,0.071338,0.592826,0.843046,0.061556,0.595581,0.627041,0.086411,0.603624,...,0.610830,0.999321,0.200949,0.798366,0.573783,0.099075,0.474813,0.730034,0.078498,0.647279
4,ADEPT05,6.6688490e-01,0.068983,0.614509,0.975962,0.054164,0.751326,0.623509,0.064267,0.627291,...,0.602847,0.991634,0.190448,0.801202,0.567557,0.126154,0.441525,0.791421,0.066190,0.727058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,MWI_CN82,5.6422101e-01,0.077352,0.523213,0.666636,0.090259,0.581154,0.528409,0.100684,0.451361,...,0.474700,0.901093,0.220515,0.680470,0.488034,0.144081,0.344001,0.621078,0.117980,0.506970
270,MWI_CN83,6.9848789e-01,0.133280,0.594502,0.743387,0.105156,0.491800,0.542317,0.154168,0.442211,...,0.526787,0.860828,0.222414,0.638332,0.536061,0.155244,0.380953,0.602585,0.118519,0.487089
271,MWI_CN84,8.3637830e-01,0.075306,0.682050,0.715348,0.075344,0.519286,0.524379,0.093592,0.488558,...,0.343942,0.769077,0.179261,0.590332,0.535888,0.157677,0.378221,0.599822,0.092429,0.508567
272,VSI41_8550,4.9683935e-01,0.090352,0.425914,0.759475,0.112112,0.489371,0.569115,0.132434,0.478665,...,0.368515,0.890930,0.280834,0.610054,0.601955,0.208697,0.393320,0.717549,0.149375,0.571504


In [43]:
ALPS_DIR = r"D:\JHAhn\files"
ALPS_FILE = f"{ALPS_DIR}\Auto_ALPS_index_19-24_M03.xlsx"
dfCN = pd.read_excel(ALPS_FILE, sheet_name='CN')
dfAD = pd.read_excel(ALPS_FILE, sheet_name='Dementia')

def drop_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.columns.str.startswith("Unnamed:")]
dfCN = drop_unnamed_columns(dfCN)
dfAD = drop_unnamed_columns(dfAD)

def drop_after_sgds(df):
    if 'SGDS' not in df.columns:
        raise KeyError("'SGDS' 컬럼을 찾을 수 없습니다.")
    keep_upto = df.columns.get_loc('SGDS')  # SGDS 포함
    return df.iloc[:, :keep_upto + 1]
dfCN = drop_after_sgds(dfCN)
dfAD = drop_after_sgds(dfAD)

print(dfCN.shape, '\n', dfAD.shape)

(58, 62) 
 (209, 62)


In [53]:
# ROI keys (df) --> Dict
roi_dict = dfROI_sub.set_index("PDI").to_dict(orient="index")

def match_roi_key(full_pid):
    for key in roi_dict:
        if key in str(full_pid):
            return key
    return None
def attach_roi_partial(df):
    # Find matching key
    df["_roi_key"] = df["PDI"].astype(str).apply(match_roi_key)
    # Filling ROI cols
    roi_cols = list(next(iter(roi_dict.values())).keys())
    for col in roi_cols:
        df[col] = df["_roi_key"].map(
            lambda k: roi_dict[k][col] if k is not None else pd.NA
        )

    return df.drop(columns="_roi_key")

dfCN_mgd = attach_roi_partial(dfCN.copy())
dfAD_mgd = attach_roi_partial(dfAD.copy())

In [54]:
dfCN_mgd

,PDI,PID,DDX,Year,Age,sex,Education_yr,LH-ALL_Dxx_proj,LH-ALL_Dyy_proj,LH-ALL_Dzz_proj,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,MWI_CN01_KNH_NODDI32,145616,NL,2019.0,70,F,16.0,0.853582,0.731372,1.259900,...,0.569473,0.826935,0.271682,0.555182,0.596481,0.212770,0.383751,0.628513,0.125133,0.503758
1,MWI_CN04_SSS_NODDI32,2716077,NL,2019.0,62,F,12.0,0.721595,0.654347,1.212313,...,0.489838,0.761443,0.233806,0.527660,0.482623,0.188198,0.294517,0.592590,0.139421,0.453941
2,MWI_CN05_LSS_NODDI32,469852,NL,2019.0,72,M,16.0,0.648850,0.534350,1.036836,...,0.544412,0.850338,0.298828,0.551536,0.560023,0.178677,0.381515,0.707167,0.123957,0.582434
3,MWI_CN06_KDS_NODDI32,2296061,NL,2019.0,72,M,6.0,0.776232,0.855603,0.955848,...,0.573242,1.072521,0.284065,0.788510,0.623203,0.149430,0.473831,0.774144,0.129967,0.644462
4,MWI_CN07_KSB_NODDI32,316024,NL,2019.0,66,M,9.0,0.760503,0.692654,1.243302,...,0.536905,0.884778,0.235095,0.649583,0.540163,0.201650,0.338564,0.664265,0.148808,0.523295
5,MWI_CN09_YKJ_NODDI32,80969,NaMCI,2019.0,65,F,12.0,0.721305,0.585058,1.168966,...,0.545306,0.986288,0.226849,0.759351,0.528989,0.166927,0.362146,0.656091,0.120433,0.535400
6,MWI_CN10_CJS_NODDI32,1147259,Other,2019.0,66,F,9.0,0.766402,0.500861,1.051554,...,0.449067,0.874900,0.364917,0.509969,0.502285,0.193763,0.308677,0.674751,0.154631,0.521590
7,MWI_CN12_LJK_NODDI32,5871888,NL,2019.0,66,F,12.0,0.809517,0.506511,1.159348,...,0.462334,0.860930,0.293146,0.567797,0.561543,0.224490,0.337122,0.663349,0.132673,0.526265
8,MWI_CN13_KH_NODDI32,6476189,SaMCI,2019.0,69,M,16.0,0.661128,0.740881,1.190856,...,0.522063,1.011569,0.237576,0.773883,0.517119,0.085755,0.431430,0.617109,0.129431,0.483453
9,MWI_CN14_PYJ_NODDI32,4341115,NaMCI,2019.0,66,F,0.5,0.667318,0.716731,1.135098,...,0.527500,1.189327,0.276960,0.912264,0.568219,0.154569,0.413899,0.629691,0.130360,0.501892


In [55]:
dfAD_mgd

,PDI,PID,DDX,Year,Age,sex,Education_yr,LH-ALL_Dxx_proj,LH-ALL_Dyy_proj,LH-ALL_Dzz_proj,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,MREPT2019AD01_KMO_NODDI32,3735478,MaMCI,2019,70.0,F,14.0,0.644891,0.617538,1.173563,...,0.574908,0.836188,0.269571,0.567105,0.589605,0.171864,0.417809,0.673767,0.123427,0.55042
1,MREPT2019AD02_JSJ_NODDI32,4381137,SaMCI,2019,68.0,F,13.0,0.711342,0.473727,1.183256,...,0.523015,0.743879,0.262853,0.481046,0.500492,0.165644,0.334925,0.598505,0.142315,0.457942
2,MREPT2019AD03_OHS_NODDI32,4452316,Other,2019,67.0,F,9.0,0.817484,0.701432,1.253393,...,0.519948,0.885777,0.26201,0.623679,0.528768,0.164828,0.364057,0.579572,0.12915,0.450615
3,MREPT2019AD04_CSY_NODDI32,4681566,Other,2019,66.0,M,12.0,0.636522,0.566616,1.020413,...,0.449734,0.751123,0.223271,0.527847,0.543459,0.2151,0.328426,0.657526,0.129952,0.529767
4,MREPT2019AD05_KJI_NODDI32,7205823,SaMCI,2019,77.0,F,6.0,0.884875,0.763448,1.324970,...,0.516597,0.906839,0.22427,0.682516,0.653884,0.16927,0.484581,0.722335,0.141361,0.584527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,EPT2024AD09_CSO_NODDI32,5212646,Other,2024,79.0,F,NaN,0.777790,0.751147,1.083035,...,0.592709,0.793156,0.242145,0.551257,0.59785,0.143183,0.454816,0.770251,0.122816,0.650285
205,EPT2024AD10_KCJ_NODDI32,7295104,Other,2024,77.0,F,NaN,0.703165,0.449837,1.133783,...,0.360986,0.740868,0.237083,0.503703,0.536423,0.194904,0.341641,0.634578,0.122864,0.507875
206,EPT2024AD11_KHJ_NODDI32,7760856,Other,2024,70.0,F,NaN,0.695299,0.652078,1.204808,...,0.459306,0.886238,0.212521,0.67383,0.583785,0.140967,0.44292,0.638426,0.118866,0.519198
207,EPT2024AD12_LDS_NODDI32,10130501,AD,2024,58.0,M,NaN,0.758654,0.879287,1.034725,...,0.650494,0.910185,0.255858,0.65449,0.624966,0.152604,0.472451,0.771421,0.119155,0.65466


In [59]:
df = dfAD_mgd
for column in df.columns:
    print(f"Col name: {column}")
    print(f"Data type: {df[column].dtype}")
    print(f"Unique values: {len(df[column].unique())}")
    print(df[column].unique())
    print('-'*40)

Col name: PDI
Data type: object
Unique values: 209
['MREPT2019AD01_KMO_NODDI32' 'MREPT2019AD02_JSJ_NODDI32'
 'MREPT2019AD03_OHS_NODDI32' 'MREPT2019AD04_CSY_NODDI32'
 'MREPT2019AD05_KJI_NODDI32' 'MREPT2019AD06_CKJ_NODDI32'
 'MREPT2019AD07_LJH_NODDI32' 'MREPT2019AD09_LKS_NODDI32'
 'MREPT2019AD10_KJL_NODDI32' 'MREPT2019AD11_CKS_NODDI32'
 'MREPT2019AD13_KHS_NODDI32' 'MREPT2019AD14_CMJ_NODDI32'
 'ADEPT01_YSJ_NODDI24' 'ADEPT02_LKY_NODDI24' 'ADEPT03_YKJ_NODDI24'
 'ADEPT04_JWS_NODDI24' 'ADEPT05_CBK_NODDI24' 'ADEPT06_PHJ_NODDI24'
 'ADEPT07_JG_NODDI24' 'ADEPT08_KMJ_NODDI24' 'ADEPT09_JCJ_NODDI24'
 'ADEPT10_YPL_NODDI24' 'ADEPT16_LJS_NODDI24' 'ADEPT18_PST_NODDI24'
 'ADEPT21_CJM_NODDI24' 'ADEPT22_KBS_NODDI24' 'ADEPT23_PYS_NODDI24'
 'ADEPT24_KJH_NODDI24' 'ADEPT25_JKR_NODDI24' 'ADEPT26_KYJ_NODDI24'
 'ADEPT27_KKS_NODDI24' 'ADEPT28_MSH_NODDI24' 'ADEPT29_LBS_NODDI24'
 'ADEPT30_WJS_NODDI24' 'ADEPT31_PTL_NODDI24' 'ADEPT32_KSW_NODDI24'
 'ADEPT33_YCY_NODDI24' 'ADEPT34_PDY_NODDI24' 'ADEPT36_JJS_NODDI24'
 'ADE

In [58]:
for df in (dfCN_mgd, dfAD_mgd):
    df.columns = (
        df.columns
          .str.replace('LH-', 'LT_', regex=False)
          .str.replace('RH-', 'RT_', regex=False)
    )
dfCN_mgd

,PDI,PID,DDX,Year,Age,sex,Education_yr,LT_ALL_Dxx_proj,LT_ALL_Dyy_proj,LT_ALL_Dzz_proj,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,MWI_CN01_KNH_NODDI32,145616,NL,2019.0,70,F,16.0,0.853582,0.731372,1.259900,...,0.569473,0.826935,0.271682,0.555182,0.596481,0.212770,0.383751,0.628513,0.125133,0.503758
1,MWI_CN04_SSS_NODDI32,2716077,NL,2019.0,62,F,12.0,0.721595,0.654347,1.212313,...,0.489838,0.761443,0.233806,0.527660,0.482623,0.188198,0.294517,0.592590,0.139421,0.453941
2,MWI_CN05_LSS_NODDI32,469852,NL,2019.0,72,M,16.0,0.648850,0.534350,1.036836,...,0.544412,0.850338,0.298828,0.551536,0.560023,0.178677,0.381515,0.707167,0.123957,0.582434
3,MWI_CN06_KDS_NODDI32,2296061,NL,2019.0,72,M,6.0,0.776232,0.855603,0.955848,...,0.573242,1.072521,0.284065,0.788510,0.623203,0.149430,0.473831,0.774144,0.129967,0.644462
4,MWI_CN07_KSB_NODDI32,316024,NL,2019.0,66,M,9.0,0.760503,0.692654,1.243302,...,0.536905,0.884778,0.235095,0.649583,0.540163,0.201650,0.338564,0.664265,0.148808,0.523295
5,MWI_CN09_YKJ_NODDI32,80969,NaMCI,2019.0,65,F,12.0,0.721305,0.585058,1.168966,...,0.545306,0.986288,0.226849,0.759351,0.528989,0.166927,0.362146,0.656091,0.120433,0.535400
6,MWI_CN10_CJS_NODDI32,1147259,Other,2019.0,66,F,9.0,0.766402,0.500861,1.051554,...,0.449067,0.874900,0.364917,0.509969,0.502285,0.193763,0.308677,0.674751,0.154631,0.521590
7,MWI_CN12_LJK_NODDI32,5871888,NL,2019.0,66,F,12.0,0.809517,0.506511,1.159348,...,0.462334,0.860930,0.293146,0.567797,0.561543,0.224490,0.337122,0.663349,0.132673,0.526265
8,MWI_CN13_KH_NODDI32,6476189,SaMCI,2019.0,69,M,16.0,0.661128,0.740881,1.190856,...,0.522063,1.011569,0.237576,0.773883,0.517119,0.085755,0.431430,0.617109,0.129431,0.483453
9,MWI_CN14_PYJ_NODDI32,4341115,NaMCI,2019.0,66,F,0.5,0.667318,0.716731,1.135098,...,0.527500,1.189327,0.276960,0.912264,0.568219,0.154569,0.413899,0.629691,0.130360,0.501892


In [61]:
for df in (dfCN_mgd, dfAD_mgd):
    df["PID"] = df["PID"].astype(str).str.zfill(8)
dfCN_mgd

,PDI,PID,DDX,Year,Age,sex,Education_yr,LT_ALL_Dxx_proj,LT_ALL_Dyy_proj,LT_ALL_Dzz_proj,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,MWI_CN01_KNH_NODDI32,00145616,NL,2019.0,70,F,16.0,0.853582,0.731372,1.259900,...,0.569473,0.826935,0.271682,0.555182,0.596481,0.212770,0.383751,0.628513,0.125133,0.503758
1,MWI_CN04_SSS_NODDI32,02716077,NL,2019.0,62,F,12.0,0.721595,0.654347,1.212313,...,0.489838,0.761443,0.233806,0.527660,0.482623,0.188198,0.294517,0.592590,0.139421,0.453941
2,MWI_CN05_LSS_NODDI32,00469852,NL,2019.0,72,M,16.0,0.648850,0.534350,1.036836,...,0.544412,0.850338,0.298828,0.551536,0.560023,0.178677,0.381515,0.707167,0.123957,0.582434
3,MWI_CN06_KDS_NODDI32,02296061,NL,2019.0,72,M,6.0,0.776232,0.855603,0.955848,...,0.573242,1.072521,0.284065,0.788510,0.623203,0.149430,0.473831,0.774144,0.129967,0.644462
4,MWI_CN07_KSB_NODDI32,00316024,NL,2019.0,66,M,9.0,0.760503,0.692654,1.243302,...,0.536905,0.884778,0.235095,0.649583,0.540163,0.201650,0.338564,0.664265,0.148808,0.523295
5,MWI_CN09_YKJ_NODDI32,00080969,NaMCI,2019.0,65,F,12.0,0.721305,0.585058,1.168966,...,0.545306,0.986288,0.226849,0.759351,0.528989,0.166927,0.362146,0.656091,0.120433,0.535400
6,MWI_CN10_CJS_NODDI32,01147259,Other,2019.0,66,F,9.0,0.766402,0.500861,1.051554,...,0.449067,0.874900,0.364917,0.509969,0.502285,0.193763,0.308677,0.674751,0.154631,0.521590
7,MWI_CN12_LJK_NODDI32,05871888,NL,2019.0,66,F,12.0,0.809517,0.506511,1.159348,...,0.462334,0.860930,0.293146,0.567797,0.561543,0.224490,0.337122,0.663349,0.132673,0.526265
8,MWI_CN13_KH_NODDI32,06476189,SaMCI,2019.0,69,M,16.0,0.661128,0.740881,1.190856,...,0.522063,1.011569,0.237576,0.773883,0.517119,0.085755,0.431430,0.617109,0.129431,0.483453
9,MWI_CN14_PYJ_NODDI32,04341115,NaMCI,2019.0,66,F,0.5,0.667318,0.716731,1.135098,...,0.527500,1.189327,0.276960,0.912264,0.568219,0.154569,0.413899,0.629691,0.130360,0.501892


In [62]:
dfAD_mgd

,PDI,PID,DDX,Year,Age,sex,Education_yr,LT_ALL_Dxx_proj,LT_ALL_Dyy_proj,LT_ALL_Dzz_proj,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,MREPT2019AD01_KMO_NODDI32,03735478,MaMCI,2019,70.0,F,14.0,0.644891,0.617538,1.173563,...,0.574908,0.836188,0.269571,0.567105,0.589605,0.171864,0.417809,0.673767,0.123427,0.55042
1,MREPT2019AD02_JSJ_NODDI32,04381137,SaMCI,2019,68.0,F,13.0,0.711342,0.473727,1.183256,...,0.523015,0.743879,0.262853,0.481046,0.500492,0.165644,0.334925,0.598505,0.142315,0.457942
2,MREPT2019AD03_OHS_NODDI32,04452316,Other,2019,67.0,F,9.0,0.817484,0.701432,1.253393,...,0.519948,0.885777,0.26201,0.623679,0.528768,0.164828,0.364057,0.579572,0.12915,0.450615
3,MREPT2019AD04_CSY_NODDI32,04681566,Other,2019,66.0,M,12.0,0.636522,0.566616,1.020413,...,0.449734,0.751123,0.223271,0.527847,0.543459,0.2151,0.328426,0.657526,0.129952,0.529767
4,MREPT2019AD05_KJI_NODDI32,07205823,SaMCI,2019,77.0,F,6.0,0.884875,0.763448,1.324970,...,0.516597,0.906839,0.22427,0.682516,0.653884,0.16927,0.484581,0.722335,0.141361,0.584527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,EPT2024AD09_CSO_NODDI32,05212646,Other,2024,79.0,F,NaN,0.777790,0.751147,1.083035,...,0.592709,0.793156,0.242145,0.551257,0.59785,0.143183,0.454816,0.770251,0.122816,0.650285
205,EPT2024AD10_KCJ_NODDI32,07295104,Other,2024,77.0,F,NaN,0.703165,0.449837,1.133783,...,0.360986,0.740868,0.237083,0.503703,0.536423,0.194904,0.341641,0.634578,0.122864,0.507875
206,EPT2024AD11_KHJ_NODDI32,07760856,Other,2024,70.0,F,NaN,0.695299,0.652078,1.204808,...,0.459306,0.886238,0.212521,0.67383,0.583785,0.140967,0.44292,0.638426,0.118866,0.519198
207,EPT2024AD12_LDS_NODDI32,10130501,AD,2024,58.0,M,NaN,0.758654,0.879287,1.034725,...,0.650494,0.910185,0.255858,0.65449,0.624966,0.152604,0.472451,0.771421,0.119155,0.65466


In [64]:
with pd.ExcelWriter(os.path.join(ALPS_DIR, "Auto_ALPS_index_19-24_0731.xlsx")
                    , engine='openpyxl') as writer:
    dfCN_mgd.to_excel(writer, sheet_name='CN',       index=False)
    dfAD_mgd.to_excel(writer, sheet_name='Dementia', index=False)